# LogFormer — Log Anomaly Detection Benchmark

**Paper:** Guo et al. (AAAI 2024) — *LogFormer: A Pre-train and Tuning Pipeline for Log Anomaly Detection*  
**Architecture:** Transformer Encoder + Adapter, pre-trained then adapter-tuned  
**Datasets:** HDFS_v1 · BGL  

---
### Kaggle Setup
1. Upload datasets vào Kaggle Dataset `logad-data`:
   - `HDFS.log` → `/kaggle/input/logad-data/hdfs/HDFS.log`
   - `anomaly_label.csv` → `/kaggle/input/logad-data/hdfs/anomaly_label.csv`
   - `BGL.log` → `/kaggle/input/logad-data/bgl/BGL.log`
2. **Enable GPU** (T4 hoặc P100)
3. Run All

**Pipeline:** Raw log → Drain parse → SentenceTransformer embed → `.npz` → `train_transformer.py` → `tune_transformer.py`

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/rubyhcm/code-papers.git'
REPO_DIR = '/kaggle/working/code-papers'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print('Repo ready:', REPO_DIR)

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q sentence-transformers scikit-learn tqdm

# LogFormer uses drain3 via its own Drain.py — no extra install needed
print('Dependencies installed.')

In [ ]:
# ── 3. Config ─────────────────────────────────────────────────────────────────
import os, sys
import torch

LOGFORMER_DIR = os.path.join(REPO_DIR, 'LogFormer')
sys.path.insert(0, LOGFORMER_DIR)

HDFS_LOG    = '/kaggle/input/logad-data/hdfs/HDFS.log'
HDFS_LABEL  = '/kaggle/input/logad-data/hdfs/anomaly_label.csv'
BGL_LOG     = '/kaggle/input/logad-data/bgl/BGL.log'
OUTPUT_DIR  = '/kaggle/working/logformer_results'
CKPT_DIR    = '/kaggle/working/logformer_checkpoints'

for d in [OUTPUT_DIR, CKPT_DIR,
          os.path.join(LOGFORMER_DIR, 'log_data'),
          os.path.join(LOGFORMER_DIR, 'parse_result'),
          os.path.join(LOGFORMER_DIR, 'preprocessed_data'),
          os.path.join(LOGFORMER_DIR, 'checkpoints'),
          os.path.join(LOGFORMER_DIR, 'result')]:
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Hyperparams (match paper defaults)
WINDOW_SIZE   = 20
PRE_EPOCHS    = 10    # pre-training
TUNE_EPOCHS   = 20    # adapter tuning
LR_PRETRAIN   = '1e-5'
LR_TUNE       = '1e-5'
ADAPTER_SIZE  = 64
NUM_LAYERS    = 1

## Step 1: Log Parsing (Drain)

In [ ]:
# ── 4. Symlink raw logs into LogFormer's expected log_data/ dir ───────────────
log_data_dir = os.path.join(LOGFORMER_DIR, 'log_data')

for src, name in [
    (HDFS_LOG,   'HDFS.log'),
    (HDFS_LABEL, 'anomaly_label.csv'),
    (BGL_LOG,    'BGL.log'),
]:
    dst = os.path.join(log_data_dir, name)
    if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'Linked: {name}')
print('log_data dir:', os.listdir(log_data_dir))

In [ ]:
# ── 5. Parse HDFS with Drain (LogFormer's parse_log.py) ──────────────────────
os.chdir(LOGFORMER_DIR)

# parse_log.py parses both HDFS and BGL; run it directly
!python parse_log.py 2>&1 | tail -10

print('Parse results:', os.listdir(os.path.join(LOGFORMER_DIR, 'parse_result')))

## Step 2: Preprocess (SentenceTransformer embeddings → .npz)

In [ ]:
# ── 6a. Preprocess BGL ────────────────────────────────────────────────────────
os.chdir(os.path.join(LOGFORMER_DIR, 'preprocess'))
!python preprocess_bgl.py 2>&1 | tail -15

print('Preprocessed data:',
      os.listdir(os.path.join(LOGFORMER_DIR, 'preprocessed_data')))

In [ ]:
# ── 6b. Preprocess HDFS ───────────────────────────────────────────────────────
os.chdir(os.path.join(LOGFORMER_DIR, 'preprocess'))
!python preprocess_hdfs.py 2>&1 | tail -15

print('Preprocessed data:',
      os.listdir(os.path.join(LOGFORMER_DIR, 'preprocessed_data')))

## Step 3: Pre-training

In [ ]:
# ── 7a. Pre-train on BGL ──────────────────────────────────────────────────────
os.chdir(LOGFORMER_DIR)
!python train_transformer.py \
    --log_name BGL \
    --window_size {WINDOW_SIZE} \
    --mode classifier \
    --num_layers {NUM_LAYERS} \
    --adapter_size {ADAPTER_SIZE} \
    --lr {LR_PRETRAIN} \
    2>&1 | tee {OUTPUT_DIR}/pretrain_bgl.log

print('BGL pre-training done.')
print('Checkpoints:', os.listdir(os.path.join(LOGFORMER_DIR, 'checkpoints')))

In [ ]:
# ── 7b. Pre-train on HDFS ─────────────────────────────────────────────────────
os.chdir(LOGFORMER_DIR)
!python train_transformer.py \
    --log_name HDFS \
    --window_size {WINDOW_SIZE} \
    --mode classifier \
    --num_layers {NUM_LAYERS} \
    --adapter_size {ADAPTER_SIZE} \
    --lr {LR_PRETRAIN} \
    2>&1 | tee {OUTPUT_DIR}/pretrain_hdfs.log

print('HDFS pre-training done.')

## Step 4: Adapter Tuning

In [ ]:
# ── 8a. Find best pre-trained checkpoint for BGL ─────────────────────────────
import glob

ckpt_dir = os.path.join(LOGFORMER_DIR, 'checkpoints')
bgl_ckpts  = sorted(glob.glob(f'{ckpt_dir}/train_BGL*best.pt'))
hdfs_ckpts = sorted(glob.glob(f'{ckpt_dir}/train_HDFS*best.pt'))

print('BGL checkpoints:', bgl_ckpts)
print('HDFS checkpoints:', hdfs_ckpts)

BGL_CKPT  = bgl_ckpts[-1]  if bgl_ckpts  else None
HDFS_CKPT = hdfs_ckpts[-1] if hdfs_ckpts else None
print(f'Using BGL ckpt : {BGL_CKPT}')
print(f'Using HDFS ckpt: {HDFS_CKPT}')

In [ ]:
# ── 8b. Tune adapter: BGL pretrained → BGL ───────────────────────────────────
os.chdir(LOGFORMER_DIR)
!python tune_transformer.py \
    --pretrained_log_name BGL \
    --load_path {BGL_CKPT} \
    --log_name BGL \
    --tune_mode adapter \
    --num_layers {NUM_LAYERS} \
    --adapter_size {ADAPTER_SIZE} \
    --lr {LR_TUNE} \
    --epoch {TUNE_EPOCHS} \
    --window_size {WINDOW_SIZE} \
    2>&1 | tee {OUTPUT_DIR}/tune_bgl.log

print('BGL adapter tuning done.')

In [ ]:
# ── 8c. Tune adapter: HDFS pretrained → HDFS ─────────────────────────────────
os.chdir(LOGFORMER_DIR)
!python tune_transformer.py \
    --pretrained_log_name HDFS \
    --load_path {HDFS_CKPT} \
    --log_name HDFS \
    --tune_mode adapter \
    --num_layers {NUM_LAYERS} \
    --adapter_size {ADAPTER_SIZE} \
    --lr {LR_TUNE} \
    --epoch {TUNE_EPOCHS} \
    --window_size {WINDOW_SIZE} \
    2>&1 | tee {OUTPUT_DIR}/tune_hdfs.log

print('HDFS adapter tuning done.')

## Collect & Save Results

In [ ]:
# ── 9. Parse result logs ──────────────────────────────────────────────────────
import re, json
import matplotlib.pyplot as plt
import numpy as np

def parse_logformer_log(path):
    """Extract last Precision/Recall/F1 from train or tune log."""
    result = {}
    with open(path) as f:
        for line in f:
            p = re.search(r'Precision:\s*([\d.]+)', line)
            r = re.search(r'Recall:\s*([\d.]+)', line)
            f1 = re.search(r'F1 score:\s*([\d.]+)', line)
            if p:  result['precision'] = float(p.group(1))
            if r:  result['recall']    = float(r.group(1))
            if f1: result['f1']        = float(f1.group(1))
    return result

results = {
    'metadata': {'model': 'LogFormer', 'paper': 'AAAI 2024',
                 'pre_epochs': PRE_EPOCHS, 'tune_epochs': TUNE_EPOCHS},
    'bgl': {
        'pretrain': parse_logformer_log(f'{OUTPUT_DIR}/pretrain_bgl.log'),
        'tuned':    parse_logformer_log(f'{OUTPUT_DIR}/tune_bgl.log'),
    },
    'hdfs': {
        'pretrain': parse_logformer_log(f'{OUTPUT_DIR}/pretrain_hdfs.log'),
        'tuned':    parse_logformer_log(f'{OUTPUT_DIR}/tune_hdfs.log'),
    },
}

out_path = f'{OUTPUT_DIR}/logformer_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

In [ ]:
# ── 10. Plot pretrain vs tuned per dataset ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('LogFormer — Pre-train vs Adapter Tuned', fontsize=13, fontweight='bold')
metric_names = ['Precision', 'Recall', 'F1']
metric_keys  = ['precision', 'recall', 'f1']
x = np.arange(len(metric_keys))

for ax, (dname, dkey) in zip(axes, [('BGL', 'bgl'), ('HDFS_v1', 'hdfs')]):
    pre   = results[dkey].get('pretrain', {})
    tuned = results[dkey].get('tuned', {})
    pre_v   = [pre.get(k, 0)   for k in metric_keys]
    tuned_v = [tuned.get(k, 0) for k in metric_keys]
    ax.bar(x - 0.2, pre_v,   0.38, label='Pre-train', color='#2196F3', alpha=0.85)
    ax.bar(x + 0.2, tuned_v, 0.38, label='Tuned',     color='#4CAF50', alpha=0.85)
    for i, (pv, tv) in enumerate(zip(pre_v, tuned_v)):
        ax.text(i - 0.2, pv + 0.002, f'{pv:.3f}', ha='center', fontsize=8)
        ax.text(i + 0.2, tv + 0.002, f'{tv:.3f}', ha='center', fontsize=8)
    ax.set_title(dname, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(metric_names)
    ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/logformer_results.png', bbox_inches='tight')
plt.show()

# Parse training curve from result/ txt files
result_txt_dir = os.path.join(LOGFORMER_DIR, 'result')
print('Result text files:', os.listdir(result_txt_dir))
print(f'All outputs saved to {OUTPUT_DIR}')

In [ ]:
# ── 11. Plot training curves from result/ txt ─────────────────────────────────
def parse_train_curve(txt_path):
    losses, f1s = [], []
    with open(txt_path) as f:
        for line in f:
            m_loss = re.search(r'loss\s+([\d.]+)', line)
            m_f1   = re.search(r'f1\s+([\d.]+)', line)
            if m_loss: losses.append(float(m_loss.group(1)))
            if m_f1:   f1s.append(float(m_f1.group(1)))
    return losses, f1s

txt_files = [f for f in os.listdir(result_txt_dir) if f.endswith('.txt')]
if txt_files:
    fig, axes = plt.subplots(1, len(txt_files), figsize=(6*len(txt_files), 3))
    if len(txt_files) == 1: axes = [axes]
    for ax, fname in zip(axes, sorted(txt_files)):
        losses, f1s = parse_train_curve(os.path.join(result_txt_dir, fname))
        if losses:
            ax2 = ax.twinx()
            ax.plot(losses[:200], color='#F44336', alpha=0.7, label='Loss')
            if f1s:
                ax2.plot(f1s[:200], color='#2196F3', linestyle='--', label='F1')
                ax2.set_ylabel('F1', color='#2196F3')
            ax.set_title(fname.replace('.txt',''), fontsize=9)
            ax.set_xlabel('Step'); ax.set_ylabel('Loss', color='#F44336')
            ax.legend(loc='upper left'); ax2.legend(loc='upper right')
            ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/logformer_curves.png', bbox_inches='tight')
    plt.show()
print('Done!')